# 🔬 FASE 1: BISTURÍ FORENSE (Primera Vuelta - Hasta el 21 de Junio)

**Operación BaBaYaga Core - Andrea Zabala Cárcamo**

> **Objetivo:** Ejecutar la Ley de Benford (2BL) sobre los consolidados municipales (Macro E-24) y la minería mesa a mesa (Micro E-14) para establecer la línea base algorítmica de la primera vuelta.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chisquare
import os

# Configuración de estilo visual BaBaYaga
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Ingesta Estructural (Macro y Micro)
Cargando los datasets correspondientes a la Primera Vuelta.

In [ ]:
# Dataset Macro (E-24 / Consolidado Municipal o Nacional)
ruta_macro = '../01_EVIDENCIA/DATA_1RA_VUELTA/AUDITORIA_NACIONAL_32_DEPARTAMENTOS_COLOMBIA.csv'
df_macro = pd.read_csv(ruta_macro, sep=',')

# Dataset Micro (Reporte Fraude Municipal vs Testigos)
ruta_micro = '../01_EVIDENCIA/DATA_1RA_VUELTA/REPORTE_FRAUDE_POR_MUNICIPIO.csv'
df_micro = pd.read_csv(ruta_micro, sep=',')

print(f"[+] Dataset Macro cargado: {df_macro.shape[0]} registros.")
print(f"[+] Dataset Micro cargado: {df_micro.shape[0]} registros.")

## 2. Generación de Matriz Maestra Comparativa (Mesa a Mesa)
Alineando con la orden táctica, se genera la Matriz Maestra que permitirá ver toda la cadena de custodia del voto para la Primera Vuelta: Claveros, Delegados, Transmisión, y E-24. Los campos de los que aún no se posea data granular quedarán en blanco (`NaN`) a la espera de la extracción completa.

In [ ]:
def generar_matriz_maestra():
    # Estructura forense exigida por Johannes
    columnas_maestras = [
        'Departamento', 'Municipio', 'Zona', 'Puesto', 'Mesa',
        'Votos_Claveros_Cepeda', 'Votos_Claveros_Espriella', 
        'Votos_Delegados_Cepeda', 'Votos_Delegados_Espriella',
        'Votos_Transmision_Cepeda', 'Votos_Transmision_Espriella',
        'Votos_E24_Cepeda', 'Votos_E24_Espriella',
        'Divergencia_E14_vs_E24', 'Hash_Archivo_Web'
    ]
    
    # Creamos un DataFrame vacío con la estructura de la Matriz Maestra
    df_maestra = pd.DataFrame(columns=columnas_maestras)
    
    # Aquí se iteraría sobre los archivos crudos (ej. PDFs extraídos) para poblar la matriz.
    # Guardamos la plantilla vacía / base para inyección de datos posteriores.
    ruta_salida = '../01_EVIDENCIA/DATA_1RA_VUELTA/MATRIZ_MAESTRA_1RA_VUELTA.csv'
    df_maestra.to_csv(ruta_salida, index=False)
    print(f"[+] Matriz Maestra estructurada y guardada en: {ruta_salida}")
    return df_maestra

matriz_maestra = generar_matriz_maestra()
matriz_maestra.head()

## 3. Motor Algorítmico: Ley de Benford (2BL - Segundo Dígito)
Evaluación de la distribución del segundo dígito sobre las tablas consolidadas.

In [ ]:
def extract_second_digit(number):
    if pd.isna(number):
        return -1
    num_str = str(abs(int(number)))
    if len(num_str) < 2:
        return -1
    return int(num_str[1])

benford_2bl_probs = [0.11968, 0.11389, 0.10882, 0.10433, 0.10031, 
                     0.09668, 0.09337, 0.09035, 0.08757, 0.08500]